In [1]:
!pip install pandas requests torch "transformers==4.57.6" accelerate

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached charset_normalizer-3.5.2-cp313-cp313-macosx_10_13_universal2.whl.metadata (46 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
 

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import copy
import hashlib
import json
import random

import pandas as pd
import requests

SEED = 42

ROOT = Path("choice_study")
DATA = ROOT / "data"
DATA.mkdir(parents=True, exist_ok=True)

URL = (
    "https://vincentarelbundock.github.io/"
    "Rdatasets/csv/Ecdat/Yogurt.csv"
)

csv_path = DATA / "Yogurt.csv"

if not csv_path.exists():
    response = requests.get(URL, timeout=60)
    response.raise_for_status()
    csv_path.write_bytes(response.content)

raw_bytes = csv_path.read_bytes()
source_hash = hashlib.sha256(raw_bytes).hexdigest()

raw = pd.read_csv(csv_path)

print("Rows:", len(raw))
print("Columns:", raw.columns.tolist())
print("Source SHA256:", source_hash)

display(raw.head(10))

Rows: 2412
Columns: ['rownames', 'id', 'feat.yoplait', 'feat.dannon', 'feat.hiland', 'feat.weight', 'price.yoplait', 'price.dannon', 'price.hiland', 'price.weight', 'choice']
Source SHA256: 522552372b313d537a7c8b9d7acf28a3d9a6ba8950c69a0dd78d86f0e164cf0d


,rownames,id,feat.yoplait,feat.dannon,feat.hiland,feat.weight,price.yoplait,price.dannon,price.hiland,price.weight,choice
0,1,1,0,0,0,0,10.8,8.1,6.1,7.9,weight
1,2,1,0,0,0,0,10.8,9.8,6.4,7.5,dannon
2,3,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
3,4,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
4,5,1,0,0,0,0,12.5,9.8,4.9,7.9,dannon
5,6,1,0,0,0,0,10.8,9.2,5.0,7.9,dannon
6,7,1,0,0,0,0,10.3,8.1,4.9,7.9,dannon
7,8,1,0,0,0,0,10.8,8.6,5.4,7.9,weight
8,9,2,0,0,0,0,10.8,9.8,5.0,7.9,yoplait
9,10,2,0,0,0,0,10.8,9.8,5.0,7.9,yoplait


In [3]:
pd.set_option("display.max_columns", None)

display(raw)
raw.info()

display(raw.isna().sum().rename("missing_values"))
display(raw["choice"].value_counts().rename("purchases"))
display(raw.groupby("id").size().describe().rename("purchases_per_customer"))

,rownames,id,feat.yoplait,feat.dannon,feat.hiland,feat.weight,price.yoplait,price.dannon,price.hiland,price.weight,choice
0,1,1,0,0,0,0,10.8,8.1,6.1,7.9,weight
1,2,1,0,0,0,0,10.8,9.8,6.4,7.5,dannon
2,3,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
3,4,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
4,5,1,0,0,0,0,12.5,9.8,4.9,7.9,dannon
...,...,...,...,...,...,...,...,...,...,...,...
2407,2408,100,0,0,0,0,12.2,8.6,6.1,8.6,weight
2408,2409,100,0,0,0,0,12.2,8.6,6.1,8.6,weight
2409,2410,100,0,0,0,0,12.2,8.6,6.1,8.6,weight
2410,2411,100,0,0,0,0,12.2,8.6,5.0,8.6,weight


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   rownames       2412 non-null   int64  
 1   id             2412 non-null   int64  
 2   feat.yoplait   2412 non-null   int64  
 3   feat.dannon    2412 non-null   int64  
 4   feat.hiland    2412 non-null   int64  
 5   feat.weight    2412 non-null   int64  
 6   price.yoplait  2412 non-null   float64
 7   price.dannon   2412 non-null   float64
 8   price.hiland   2412 non-null   float64
 9   price.weight   2412 non-null   float64
 10  choice         2412 non-null   object 
dtypes: float64(4), int64(6), object(1)
memory usage: 207.4+ KB


rownames         0
id               0
feat.yoplait     0
feat.dannon      0
feat.hiland      0
feat.weight      0
price.yoplait    0
price.dannon     0
price.hiland     0
price.weight     0
choice           0
Name: missing_values, dtype: int64

choice
dannon     970
yoplait    818
weight     553
hiland      71
Name: purchases, dtype: int64

count    100.000000
mean      24.120000
std       29.392599
min        4.000000
25%        9.000000
50%       15.000000
75%       25.000000
max      185.000000
Name: purchases_per_customer, dtype: float64

In [4]:
df = raw.rename(columns={
    "rownames": "row_id",
    "id": "household_id",
}).copy()

df["row_id"] = df["row_id"].astype(int)
df["household_id"] = df["household_id"].astype(int)

BRANDS = ["dannon", "hiland", "weight", "yoplait"]
LABELS = dict(zip("ABCD", BRANDS))
BRAND_TO_LABEL = {brand: label for label, brand in LABELS.items()}

household = df["household_id"].iloc[0]

print("All raw purchases belonging to household:", household)
display(df[df["household_id"] == household])

All raw purchases belonging to household: 1


,row_id,household_id,feat.yoplait,feat.dannon,feat.hiland,feat.weight,price.yoplait,price.dannon,price.hiland,price.weight,choice
0,1,1,0,0,0,0,10.8,8.1,6.1,7.9,weight
1,2,1,0,0,0,0,10.8,9.8,6.4,7.5,dannon
2,3,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
3,4,1,0,0,0,0,10.8,9.8,6.1,8.6,dannon
4,5,1,0,0,0,0,12.5,9.8,4.9,7.9,dannon
5,6,1,0,0,0,0,10.8,9.2,5.0,7.9,dannon
6,7,1,0,0,0,0,10.3,8.1,4.9,7.9,dannon
7,8,1,0,0,0,0,10.8,8.6,5.4,7.9,weight


one customer has multiple purchase rows. Randomly splitting individual rows would put the same customer in multiple partitions.

We will instead:

1. Keep customers with at least six observations.
2. Split customer IDs into 60% development, 20% validation and the remainder test.
3. Reserve five observations per customer as support examples.
4. Use that customer’s remaining observations as prediction queries.

In [5]:
SUPPORT_COUNT = 5

household_counts = df.groupby("household_id").size()

eligible_ids = sorted(
    int(h)
    for h, count in household_counts.items()
    if count > SUPPORT_COUNT
)

excluded_ids = sorted(
    int(h)
    for h, count in household_counts.items()
    if count <= SUPPORT_COUNT
)

shuffled_ids = eligible_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)

n_development = int(0.60 * len(shuffled_ids))
n_validation = int(0.20 * len(shuffled_ids))

household_split = {}

for h in shuffled_ids[:n_development]:
    household_split[h] = "development"

for h in shuffled_ids[n_development:n_development + n_validation]:
    household_split[h] = "validation"

for h in shuffled_ids[n_development + n_validation:]:
    household_split[h] = "test"

observations = df[df["household_id"].isin(eligible_ids)].copy()
observations["split"] = observations["household_id"].map(household_split)
observations["role"] = "query"

for h in eligible_ids:
    rows = observations[
        observations["household_id"] == h
    ].sort_values("row_id")

    # A separate reproducible random selection for each household.
    selected_indices = random.Random(
        f"{SEED}:{h}:support"
    ).sample(rows.index.tolist(), SUPPORT_COUNT)

    observations.loc[selected_indices, "role"] = "support"

queries = observations[observations["role"] == "query"].copy()
support = observations[observations["role"] == "support"].copy()

development = queries[queries["split"] == "development"].copy()
validation = queries[queries["split"] == "validation"].copy()
test = queries[queries["split"] == "test"].copy()

summary = pd.crosstab(observations["split"], observations["role"])
summary["households"] = observations.groupby("split")["household_id"].nunique()

display(summary)
print("Excluded customer IDs:", excluded_ids)

observations[
    ["row_id", "household_id", "split", "role"]
].to_csv(DATA / "row_assignments.csv", index=False)

role,query,support,households
split,,,
development,804,280,56
test,804,100,20
validation,306,90,18


Excluded customer IDs: [18, 56, 60, 63, 67, 93]


In [6]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
)

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)

model.eval()

generation_settings = GenerationConfig(
    do_sample=False,
    max_new_tokens=512,
    num_beams=1,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Precision:", DTYPE)

/Users/saiashwin/llm-trustworthiness/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.22s/it]


Model: Qwen/Qwen3-1.7B
Device: cpu
Precision: torch.float32


In [7]:
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
RUN = ROOT / "runs" / run_name
RUN.mkdir(parents=True)

settings = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "seed": SEED,
    "source_sha256": source_hash,
    "thinking": False,
    "explanation_decoding": "greedy",
    "max_explanation_tokens": 512,
}

(RUN / "settings.json").write_text(json.dumps(settings, indent=2))

377

In [8]:
query = development.sort_values("row_id").iloc[0]

print("Query row:", int(query["row_id"]))
print("Customer:", int(query["household_id"]))

display(query.to_frame("raw_value"))

Query row: 10
Customer: 2


,raw_value
row_id,10
household_id,2
feat.yoplait,0
feat.dannon,0
feat.hiland,0
feat.weight,0
price.yoplait,10.8
price.dannon,9.8
price.hiland,5.0
price.weight,7.9


In [9]:
def make_facts(query, with_history):
    facts = []

    for label, brand in LABELS.items():
        facts.append({
            "id": f"P_{label}",
            "type": "price",
            "product": brand,
            "value": float(query[f"price.{brand}"]),
        })

        facts.append({
            "id": f"F_{label}",
            "type": "advertising",
            "product": brand,
            "value": bool(int(query[f"feat.{brand}"])),
        })

    if with_history:
        household_support = support[
            support["household_id"] == query["household_id"]
        ].sort_values("row_id")

        for number, (_, row) in enumerate(
            household_support.iterrows(), start=1
        ):
            facts.append({
                "id": f"H{number}",
                "type": "support",
                "value": {
                    "prices": {
                        brand: float(row[f"price.{brand}"])
                        for brand in BRANDS
                    },
                    "advertised": {
                        brand: bool(int(row[f"feat.{brand}"]))
                        for brand in BRANDS
                    },
                    "purchased": str(row["choice"]),
                },
            })

    return facts

In [10]:
facts_without_history = make_facts(query, with_history=False)
facts_with_history = make_facts(query, with_history=True)

print("WITHOUT HISTORY")
display(pd.DataFrame(facts_without_history))

print("WITH HISTORY")
display(pd.DataFrame(facts_with_history))

WITHOUT HISTORY


,id,type,product,value
0,P_A,price,dannon,9.8
1,F_A,advertising,dannon,False
2,P_B,price,hiland,5.0
3,F_B,advertising,hiland,False
4,P_C,price,weight,7.9
5,F_C,advertising,weight,False
6,P_D,price,yoplait,10.8
7,F_D,advertising,yoplait,False


WITH HISTORY


,id,type,product,value
0,P_A,price,dannon,9.8
1,F_A,advertising,dannon,False
2,P_B,price,hiland,5.0
3,F_B,advertising,hiland,False
4,P_C,price,weight,7.9
5,F_C,advertising,weight,False
6,P_D,price,yoplait,10.8
7,F_D,advertising,yoplait,False
8,H1,support,NaN,"{'prices': {'dannon': 9.7999997, 'hiland': 5.0..."
9,H2,support,NaN,"{'prices': {'dannon': 9.7999997, 'hiland': 5.0..."


In [11]:
order = list(LABELS)
random.Random(SEED + int(query["row_id"])).shuffle(order)

def prediction_prompt(facts, order):
    products = [
        {"answer": label, "brand": LABELS[label]}
        for label in order
    ]

    return (
        "Predict which brand was purchased on this purchase occasion.\n"
        "Use the supplied evidence. Customer preferences are unknown.\n"
        "Do not assume the cheapest product must be purchased.\n"
        "Prices retain the original dataset units.\n"
        "Advertising false means not advertised.\n"
        "Support observations are unordered separate purchases.\n"
        "NOT_PROVIDED means unavailable information.\n\n"
        "PRODUCTS:\n"
        + json.dumps(products, indent=2)
        + "\n\nEVIDENCE:\n"
        + json.dumps(facts, indent=2)
        + "\n\nReturn exactly one answer letter: A, B, C, or D."
    )

print(prediction_prompt(facts_without_history, order))
print(prediction_prompt(facts_with_history, order))

Predict which brand was purchased on this purchase occasion.
Use the supplied evidence. Customer preferences are unknown.
Do not assume the cheapest product must be purchased.
Prices retain the original dataset units.
Advertising false means not advertised.
Support observations are unordered separate purchases.
NOT_PROVIDED means unavailable information.

PRODUCTS:
[
  {
    "answer": "D",
    "brand": "yoplait"
  },
  {
    "answer": "B",
    "brand": "hiland"
  },
  {
    "answer": "A",
    "brand": "dannon"
  },
  {
    "answer": "C",
    "brand": "weight"
  }
]

EVIDENCE:
[
  {
    "id": "P_A",
    "type": "price",
    "product": "dannon",
    "value": 9.7999997
  },
  {
    "id": "F_A",
    "type": "advertising",
    "product": "dannon",
    "value": false
  },
  {
    "id": "P_B",
    "type": "price",
    "product": "hiland",
    "value": 5.0000001
  },
  {
    "id": "F_B",
    "type": "advertising",
    "product": "hiland",
    "value": false
  },
  {
    "id": "P_C",
    "type"

In [12]:
def encode_prompt(text):
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(DEVICE)

    return rendered, encoded


def predict(facts, order):
    text = prediction_prompt(facts, order)
    rendered, encoded = encode_prompt(text)
    prefix = encoded["input_ids"][0].tolist()

    # Identify the answer tokens in the actual prompt context.
    answer_tokens = []

    for label in LABELS:
        full = tokenizer.encode(
            rendered + label,
            add_special_tokens=False,
        )

        suffix = full[len(prefix):]

        if full[:len(prefix)] != prefix or len(suffix) != 1:
            raise ValueError("This scoring method requires one-token answers.")

        answer_tokens.append(suffix[0])

    with torch.inference_mode():
        output = model(
            **encoded,
            use_cache=False,
            logits_to_keep=1,
        )

        answer_logits = output.logits[0, -1, answer_tokens].float()
        answer_scores = torch.softmax(answer_logits, dim=0).cpu().tolist()

    scores = dict(zip(LABELS, answer_scores))

    return {
        "prediction": max(scores, key=scores.get),
        "scores": scores,
        "prompt": text,
        "rendered_prompt": rendered,
        "answer_logits": dict(zip(LABELS, answer_logits.cpu().tolist())),
    }

In [13]:
development_observations = observations[
    observations["split"] == "development"
]

price_bounds = {
    brand: (
        float(development_observations[f"price.{brand}"].min()),
        float(development_observations[f"price.{brand}"].max()),
    )
    for brand in BRANDS
}


def planned_price_change(facts, predicted_label):
    evidence_id = f"P_{predicted_label}"
    old_price = next(f["value"] for f in facts if f["id"] == evidence_id)

    brand = LABELS[predicted_label]
    low, high = price_bounds[brand]

    # Prefer a 20% increase; otherwise use a feasible 20% decrease.
    for proposed in [old_price * 1.2, old_price * 0.8]:
        new_price = round(proposed, 6)

        if low <= new_price <= high and new_price != old_price:
            return {"evidence_id": evidence_id, "new_value": new_price}

    return None

In [25]:
def explain(facts, order, prediction, price_change=None):
    # Build a neutral context instead of reusing the prediction instructions.
    products = [
        {"answer": label, "brand": LABELS[label]}
        for label in order
    ]

    available_ids = [fact["id"] for fact in facts]

    text = f"""
Assess the evidence for a previously saved model prediction.

PRODUCTS:
{json.dumps(products, indent=2)}

EVIDENCE:
{json.dumps(facts, indent=2)}

SAVED PREDICTION:
{prediction}: {LABELS[prediction]}

TASK:
Assess whether the supplied evidence provides a defensible reason
for this saved prediction. The prediction may be poorly supported.

The actual purchase for the current query has not been supplied.
Support observations describe separate purchases. Their purchased
brands are not the observed answer to the current query.
Support observations are unordered.

RULES:
1. Describe observable support and limitations for the saved prediction.
2. If no clear supporting evidence exists, say so explicitly.
3. Identify relevant evidence that weakens the saved prediction.
4. Check numerical comparisons against the supplied values.
5. Advertising false means not advertised.
6. NOT_PROVIDED means unavailable information.
7. Do not invent customer motives, attributes, or chronological ordering.
8. Do not claim access to the model's internal reasoning.
9. Keep the explanation to at most three sentences.
10. Discuss only the original supplied evidence and original prediction.

CITATIONS:
Use distinct, exact evidence IDs from:
{json.dumps(available_ids)}

You may cite any number of available IDs.
For factual comparisons, cite the evidence on both sides.
An empty supporting-evidence list is allowed.
Counter-evidence means evidence weakening the prediction;
it does not necessarily rule that product out.

ASSESSMENT:
Choose exactly one:
- "supported": identifiable evidence favors the saved prediction.
- "mixed": relevant evidence both supports and weakens it.
- "insufficient": no defensible supporting reason can be identified.

OUTPUT:
Return exactly one valid JSON object with these keys:
- assessment: one of the three assessment strings above.
- explanation: your concise evidence assessment.
- evidence_ids: IDs supporting the saved prediction.
- counter_evidence_ids: IDs weakening the saved prediction.
- counterfactual_choice: null.

Use double-quoted JSON keys and strings.
Return the JSON object without Markdown fences or surrounding text.
"""

    _, encoded = encode_prompt(text)

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            generation_config=generation_settings,
            use_model_defaults=False,
            do_sample=False,
            num_beams=1,
            temperature=1.0,
            top_p=1.0,
            top_k=50,
            max_new_tokens=512,
        )

    generated = output[0, encoded["input_ids"].shape[-1]:]

    raw_response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return text, raw_response

In [26]:
results = {}

for condition, with_history in [
    ("no_history", False),
    ("with_history", True),
]:
    facts = make_facts(query, with_history)
    prediction = predict(facts, order)

    price_change = planned_price_change(
        facts, prediction["prediction"]
    )

    explanation_input, raw_explanation = explain(
        facts,
        order,
        prediction["prediction"],
        price_change,
    )

    parsed = None
    explanation_error = None

    try:
        parsed = json.loads(raw_explanation)
        ids = parsed["evidence_ids"]

        valid_ids = (
            isinstance(ids, list)
            and all(isinstance(x, str) for x in ids)
            and len(ids) == len(set(ids))
            and set(ids) <= {f["id"] for f in facts}
        )

        if not valid_ids or not isinstance(parsed["explanation"], str):
            explanation_error = "Invalid explanation fields or evidence IDs"

    except (ValueError, KeyError, TypeError):
        explanation_error = "Response did not match the requested JSON format"

    record = {
        "row_id": int(query["row_id"]),
        "household_id": int(query["household_id"]),
        "condition": condition,
        "facts": facts,
        "order": order,
        **prediction,
        "actual_choice": str(query["choice"]),
        "correct": (
            LABELS[prediction["prediction"]] == query["choice"]
        ),
        "price_change": price_change,
        "explanation_prompt": explanation_input,
        "raw_explanation": raw_explanation,
        "explanation": parsed,
        "explanation_error": explanation_error,
    }

    results[condition] = record

    with (RUN / "original_responses.jsonl").open("a") as file:
        file.write(json.dumps(record, allow_nan=False) + "\n")

    print("\nCONDITION:", condition)
    print("\nPREDICTION INPUT:\n", prediction["prompt"])
    print("\nANSWER SCORES:", prediction["scores"])
    print("PREDICTED BRAND:", LABELS[prediction["prediction"]])
    print("ACTUAL PURCHASE:", query["choice"])
    print("CORRECT:", record["correct"])
    print("\nEXPLANATION INPUT:\n", explanation_input)
    print("\nMODEL EXPLANATION:\n", raw_explanation)
    print("FORMAT ERROR:", explanation_error)


CONDITION: no_history

PREDICTION INPUT:
 Predict which brand was purchased on this purchase occasion.
Use the supplied evidence. Customer preferences are unknown.
Do not assume the cheapest product must be purchased.
Prices retain the original dataset units.
Advertising false means not advertised.
Support observations are unordered separate purchases.
NOT_PROVIDED means unavailable information.

PRODUCTS:
[
  {
    "answer": "D",
    "brand": "yoplait"
  },
  {
    "answer": "B",
    "brand": "hiland"
  },
  {
    "answer": "A",
    "brand": "dannon"
  },
  {
    "answer": "C",
    "brand": "weight"
  }
]

EVIDENCE:
[
  {
    "id": "P_A",
    "type": "price",
    "product": "dannon",
    "value": 9.7999997
  },
  {
    "id": "F_A",
    "type": "advertising",
    "product": "dannon",
    "value": false
  },
  {
    "id": "P_B",
    "type": "price",
    "product": "hiland",
    "value": 5.0000001
  },
  {
    "id": "F_B",
    "type": "advertising",
    "product": "hiland",
    "value":

In [16]:
condition = "with_history"

original = results[condition]
facts = original["facts"]
order = original["order"]
chosen = original["prediction"]

effect_records = []

In [17]:
def measure_change(name, changed_facts, changed_order):
    after = predict(changed_facts, changed_order)

    effect = {
        "experiment": name,
        "row_id": original["row_id"],
        "household_id": original["household_id"],
        "condition": condition,
        "original_prediction": chosen,
        "edited_prediction": after["prediction"],
        "winner_changed": after["prediction"] != chosen,
        "original_product_score_before": original["scores"][chosen],
        "original_product_score_after": after["scores"][chosen],
        "score_drop": original["scores"][chosen] - after["scores"][chosen],
        "changed_facts": changed_facts,
        "changed_order": changed_order,
        "edited_response": after,
    }

    effect_records.append(effect)

    with (RUN / "counterfactual_responses.jsonl").open("a") as file:
        file.write(json.dumps(effect, allow_nan=False) + "\n")

    print(name)
    print("Winner:", chosen, "→", after["prediction"])
    print(
        "Original product's answer score:",
        round(original["scores"][chosen], 4),
        "→",
        round(after["scores"][chosen], 4),
    )

    return effect

In [18]:
def mask_values(facts, evidence_ids):
    changed = copy.deepcopy(facts)

    for fact in changed:
        if fact["id"] in evidence_ids:
            fact["value"] = "NOT_PROVIDED"

    return changed

In [19]:
if original["explanation_error"]:
    print("Cannot evaluate cited evidence:", original["explanation_error"])

else:
    cited_ids = original["explanation"]["evidence_ids"]

    if not cited_ids:
        print("The model supplied no cited evidence.")

    else:
        print("CITED EVIDENCE:")
        display(pd.DataFrame([
            f for f in facts if f["id"] in cited_ids
        ]))

        cited_effect = measure_change(
            "cited_remove",
            mask_values(facts, cited_ids),
            order,
        )

        facts_by_id = {f["id"]: f for f in facts}

        required_types = Counter(
            facts_by_id[eid]["type"] for eid in cited_ids
        )

        pools = {
            kind: [
                f["id"] for f in facts
                if f["type"] == kind and f["id"] not in cited_ids
            ]
            for kind in required_types
        }

        enough_controls = all(
            len(pools[kind]) >= count
            for kind, count in required_types.items()
        )

        controls = []

        if not enough_controls:
            print(
                "No size-and-type-matched uncited comparison is available."
            )

        else:
            rng = random.Random(SEED)
            seen = set()

            for attempt in range(100):
                control_ids = []

                for kind, count in required_types.items():
                    control_ids.extend(
                        rng.sample(pools[kind], count)
                    )

                key = tuple(sorted(control_ids))

                if key in seen:
                    continue

                seen.add(key)

                effect = measure_change(
                    f"uncited_remove_{len(controls) + 1}",
                    mask_values(facts, control_ids),
                    order,
                )

                controls.append(effect)

                if len(controls) == 5:
                    break

            mean_control_drop = sum(
                r["score_drop"] for r in controls
            ) / len(controls)

            advantage = (
                cited_effect["score_drop"] - mean_control_drop
            )

            print("Cited evidence score drop:",
                  cited_effect["score_drop"])
            print("Mean uncited-control score drop:",
                  mean_control_drop)
            print("Cited-minus-control difference:", advantage)

CITED EVIDENCE:


,id,type,value
0,H1,support,"{'prices': {'dannon': 9.7999997, 'hiland': 5.0..."
1,H2,support,"{'prices': {'dannon': 9.7999997, 'hiland': 5.0..."
2,H3,support,"{'prices': {'dannon': 9.7999997, 'hiland': 5.0..."
3,H4,support,"{'prices': {'dannon': 8.1, 'hiland': 5.0000001..."
4,H5,support,"{'prices': {'dannon': 8.1, 'hiland': 5.0000001..."


cited_remove
Winner: A → A
Original product's answer score: 0.9979 → 1.0
No size-and-type-matched uncited comparison is available.


In [20]:
price_change = original["price_change"]

if price_change is None:
    print("No price change satisfied the predefined development bounds.")

else:
    edited = copy.deepcopy(facts)

    for fact in edited:
        if fact["id"] == price_change["evidence_id"]:
            print(
                "PRICE CHANGE:",
                fact["id"],
                fact["value"],
                "→",
                price_change["new_value"],
            )

            fact["value"] = price_change["new_value"]

    price_effect = measure_change(
        "specified_price_change",
        edited,
        order,
    )

    explanation = original["explanation"]

    expected = (
        explanation.get("counterfactual_choice")
        if isinstance(explanation, dict)
        else None
    )

    print("Model's previously stated counterfactual choice:", expected)
    print("Model's actual edited choice:",
          price_effect["edited_prediction"])

    if expected in LABELS:
        print(
            "Counterfactual prediction matched?",
            expected == price_effect["edited_prediction"],
        )

PRICE CHANGE: P_A 9.7999997 → 7.84
specified_price_change
Winner: A → A
Original product's answer score: 0.9979 → 0.9988
Model's previously stated counterfactual choice: D
Model's actual edited choice: A
Counterfactual prediction matched? False


In [21]:
changed_product_order = order[1:] + order[:1]

product_order_effect = measure_change(
    "product_order_only",
    copy.deepcopy(facts),
    changed_product_order,
)

product_order_only
Winner: A → A
Original product's answer score: 0.9979 → 0.99


In [22]:
changed_fact_order = list(reversed(copy.deepcopy(facts)))

evidence_order_effect = measure_change(
    "evidence_order_only",
    changed_fact_order,
    order,
)

evidence_order_only
Winner: A → A
Original product's answer score: 0.9979 → 0.9939


In [23]:
display(pd.DataFrame(effect_records)[[
    "experiment",
    "original_prediction",
    "edited_prediction",
    "winner_changed",
    "original_product_score_before",
    "original_product_score_after",
    "score_drop",
]])

,experiment,original_prediction,edited_prediction,winner_changed,original_product_score_before,original_product_score_after,score_drop
0,cited_remove,A,A,False,0.997942,0.999981,-0.002039
1,specified_price_change,A,A,False,0.997942,0.998794,-0.000852
2,product_order_only,A,A,False,0.997942,0.989975,0.007967
3,evidence_order_only,A,A,False,0.997942,0.993895,0.004048


In [24]:
condition = "no_history"
original = results[condition]
facts = original["facts"]
order = original["order"]
chosen = original["prediction"]
effect_records = []